# PDE Lambda Sweep Analysis

This notebook rewrites `runs/run_pde_lambda_sweep.py` into analysis-friendly sections so you can inspect configs, run individual cells, and explore artifacts without jumping through a monolithic script.


## 1. Setup and Imports

- Adds the repository root to `sys.path` so notebook execution is robust.
- Imports the same training, extraction, feature-metric, and plotting helpers used by the run script.


In [9]:
from __future__ import annotations

import csv
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "prog").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from prog.mlps import SirenMLP, EQL
from prog.trainer import TrainerConfig, PDETrainer
from Datasets.data.processed.burg_gen.burg_gen import BurgersDatasetConfig, build_dataset_from_burgers
from utils.derivative_utils import evaluate_primitive_feature_metrics
from utils.extract_pde_ls import extract_pde_ls
from utils.feature_plotting import save_primitive_feature_overlays
from utils.data_prep_utils import PDETrainDataset

print(f"Using repo root: {REPO_ROOT}")
 

Using repo root: c:\Users\sami\OneDrive\Documents\PDENets


## 2. Sweep Configuration

Edit these values before running the sweep cells below.


In [ ]:
LAMBDAS = [0.1, 0.3, 0.5, 0.7]
NUM_SEEDS = 1
EPOCHS = 100
BATCH_SIZE = 200
LR = 1e-3
DEVICE = "cpu"
OUT_DIR = REPO_ROOT / "run_results" / "pde_lambda_sweep_notebook"
NOISE_LEVEL = 0.05
SELECTED_DERIVS = ("u", "u_x", "u_xx")

cfg_kwargs = {
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "epochs": EPOCHS,
    "device": DEVICE,
    "selected_derivs": SELECTED_DERIVS,
}
data_kwargs = {
    "noise_level": NOISE_LEVEL,
    "stride_t": 1,
    "stride_x": 1
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
print({
    "lambdas": LAMBDAS,
    "num_seeds": NUM_SEEDS,
    "epochs": EPOCHS,
    "device": DEVICE,
    "out_dir": str(OUT_DIR),
})


{'lambdas': [0.1, 0.3, 0.5, 0.7], 'num_seeds': 1, 'epochs': 100, 'device': 'cpu', 'out_dir': 'c:\\Users\\sami\\OneDrive\\Documents\\PDENets\\run_results\\pde_lambda_sweep_notebook'}


## 3. Helper Functions

These mirror the script helpers, but stay local to the notebook so you can tweak and rerun them independently.


In [60]:
def monomial_to_term(monomial: tuple[str, ...]) -> str:
    return "*".join(monomial) if monomial else "1"


def save_pde_extraction(extraction: dict, output_dir: str | Path, method_name: str = "least_squares") -> Path:
    pde_dir = Path(output_dir) / "pde_outputs" / method_name
    pde_dir.mkdir(parents=True, exist_ok=True)

    coeffs = np.asarray(extraction["coeffs"], dtype=float)
    names = extraction["names"]
    residuals = np.asarray(extraction.get("residuals", []), dtype=float).tolist()
    singular_values = np.asarray(extraction.get("singular_values", []), dtype=float).tolist()

    with open(pde_dir / "pde.json", "w") as f:
        json.dump({
            "method": method_name,
            "feature_names": names,
            "coefficients": coeffs.tolist(),
            "residuals": residuals,
            "rank": int(extraction.get("rank", -1)),
            "singular_values": singular_values,
        }, f, indent=2)

    with open(pde_dir / "pde.txt", "w") as f:
        f.write("PDE Extraction (Least-Squares)\n" + "=" * 40 + "\n")
        for name, coeff in zip(names, coeffs):
            f.write(f"{name:12s}: {coeff:12.6e}\n")

    with open(pde_dir / "coefficients.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["feature_name", "coefficient"])
        writer.writeheader()
        for name, coeff in zip(names, coeffs):
            writer.writerow({"feature_name": name, "coefficient": float(coeff)})

    with open(pde_dir / "diagnostics.json", "w") as f:
        json.dump({
            "rank": int(extraction.get("rank", -1)),
            "singular_values": singular_values,
            "residuals": residuals,
        }, f, indent=2)

    return pde_dir


## 4. Single-Run Execution

Run one `(lambda_pde, seed)` pair end-to-end and emit the same artifacts as the script.


In [66]:
def run_single(cfg_kwargs: dict, data_kwargs: dict, lambda_pde: float, seed: int, base_out_dir: str | Path) -> dict:
    seed_dir = Path(base_out_dir) / "burgers" / f"lambda_{lambda_pde:.6f}".rstrip("0").rstrip(".") / f"seed_{seed:03d}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    data_cfg = dict(data_kwargs)
    data_cfg["seed"] = seed
    bcfg = BurgersDatasetConfig(**data_cfg)
    t_s, x_s, y_clean, y_noisy, _ = build_dataset_from_burgers(bcfg)
    t_s = np.unique(t_s)
    x_s = np.unique(x_s)
    y_clean = y_clean.reshape(t_s.shape[0], x_s.shape[0])
    print(t_s, x_s.shape, y_clean.shape)

    dataset = PDETrainDataset(
        t_grid=t_s,
        x_grid=x_s,
        u_grid=y_clean,
    )
    loader = DataLoader(dataset, batch_size=cfg_kwargs.get("batch_size", 200), shuffle=True)
    print(f"num_batches={len(loader)}")
    for batch in loader:
        t_b, x_b, u_noisy_b = batch
        print(f"batch shapes: t={t_b.shape}, x={x_b.shape}, u_noisy={u_noisy_b.shape}")
        break
    u_model = SirenMLP(
        hidden_size=cfg_kwargs.get("hidden_size", 64),
        hidden_layers=cfg_kwargs.get("hidden_layers", 3),
    )
    selected_derivs = tuple(cfg_kwargs.get("selected_derivs", ("u", "u_x", "u_xx")))
    v_model = EQL(in_dim=len(selected_derivs))

    cfg = TrainerConfig(lr=cfg_kwargs.get("lr", 1e-3))
    cfg.lambda_pde = float(lambda_pde)
    cfg.lambda_data = float(cfg_kwargs.get("lambda_data", 1.0))
    cfg.lambda_reg = float(cfg_kwargs.get("lambda_reg", 1e-3))
    cfg.lambda_tv = float(cfg_kwargs.get("lambda_tv", 0.0))
    cfg.selected_derivs = selected_derivs
    cfg.device = torch.device(cfg_kwargs.get("device", "cpu"))
    setattr(cfg, "feature_normalize", True)

    trainer = PDETrainer(u_model, v_model, cfg)
    epochs = int(cfg_kwargs.get("epochs", 200))

    with open(seed_dir / "config.json", "w") as f:
        json.dump({
            "lambda_pde": float(lambda_pde),
            "seed": int(seed),
            "epochs": int(epochs),
            "batch_size": int(cfg_kwargs.get("batch_size", 200)),
            "lr": float(cfg_kwargs.get("lr", 1e-3)),
            "lambda_data": float(cfg_kwargs.get("lambda_data", 1.0)),
            "lambda_reg": float(cfg_kwargs.get("lambda_reg", 1e-3)),
            "lambda_tv": float(cfg_kwargs.get("lambda_tv", 0.0)),
            "selected_derivs": list(selected_derivs),
            "noise_level": float(data_kwargs.get("noise_level", 0.05)),
        }, f, indent=2)

    loss_history = []
    for epoch in range(epochs):
        for batch in loader:
            t_b, x_b, u_noisy_b = batch
            metrics = trainer.step(t_b, x_b, u_noisy_b, u_noisy_b)
            loss_history.append(metrics)

    if loss_history:
        with open(seed_dir / "loss_history.csv", "w", newline="") as f:
            fieldnames = list(loss_history[0].keys())
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for entry in loss_history:
                row = {}
                for k, v in entry.items():
                    row[k] = "|".join(map(str, v)) if isinstance(v, (list, tuple)) else str(v)
                writer.writerow(row)

    t_s, x_s = np.meshgrid(t_s, x_s, indexing="ij")
    extraction = extract_pde_ls(u_model, t_s, x_s, selected_derivs, device=str(cfg.device))
    save_pde_extraction(extraction, seed_dir, method_name="least_squares")

    eql_products = v_model.product_coefficients(feature_names=list(selected_derivs), include_readout=True)
    eql_readout_poly = eql_products.get(f"p{len(v_model.linears) - 1}", {}) if len(v_model.linears) > 0 else {}
    eql_terms_and_coeffs = sorted(
        ((monomial_to_term(monomial), float(coeff)) for monomial, coeff in eql_readout_poly.items()),
        key=lambda item: item[0],
    )

    feature_eval = evaluate_primitive_feature_metrics(
        u_model,
        t_s,
        x_s,
        y_clean,
        selected_derivs,
        device=str(cfg.device),
    )
    save_primitive_feature_overlays(
        u_model,
        t_s,
        x_s,
        y_clean,
        selected_derivs,
        output_dir=seed_dir / "feature_overlays",
        device=str(cfg.device),
    )

    final_metrics = loss_history[-1] if loss_history else {}
    summary = {
        "lambda_pde": float(lambda_pde),
        "seed": int(seed),
        "final_loss": float(final_metrics.get("loss", float("nan"))),
        "final_loss_data": float(final_metrics.get("loss_data", float("nan"))),
        "final_loss_pde": float(final_metrics.get("loss_pde", float("nan"))),
        "l1": float(final_metrics.get("l1", float("nan"))),
        "loss_tv": float(final_metrics.get("loss_tv", float("nan"))),
        "pde_method": "least_squares",
        "feature_names": extraction["names"],
        "coefficients": extraction["coeffs"].tolist(),
        "eql_method": "eql",
        "eql_feature_names": list(selected_derivs),
        "eql_readout_terms": [term for term, _ in eql_terms_and_coeffs],
        "eql_readout_coeffs": [coeff for _, coeff in eql_terms_and_coeffs],
        "eql_readout_dim": int(v_model.readout.in_features),
        "num_samples": int(len(t_s)),
    }
    summary.update(feature_eval["summary_flat"])

    with open(seed_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
 
    return summary


## 5. Sweep Driver

Run all requested lambdas and seeds. This is the notebook equivalent of the script `main()` loop.


In [67]:
all_summaries = []
for lam in LAMBDAS:
    for seed in range(NUM_SEEDS):
        print(f"Running lambda_pde={lam:.6f}, seed={seed:03d} ...")
        all_summaries.append(run_single(cfg_kwargs, data_kwargs, lam, seed, OUT_DIR))

print(f"Completed {len(all_summaries)} runs.")


Running lambda_pde=0.100000, seed=000 ...
[0.    0.126 0.254 0.382 0.51  0.638 0.766 0.894] (16,) (8, 16)
num_batches=1
batch shapes: t=torch.Size([128, 1]), x=torch.Size([128, 1]), u_noisy=torch.Size([128, 1])
Running lambda_pde=0.300000, seed=000 ...
[0.    0.126 0.254 0.382 0.51  0.638 0.766 0.894] (16,) (8, 16)
num_batches=1
batch shapes: t=torch.Size([128, 1]), x=torch.Size([128, 1]), u_noisy=torch.Size([128, 1])


KeyboardInterrupt: 

## 6. Aggregate Sweep Results

Build a dataset-level `summary.csv` and a quick in-notebook table for inspection.


In [6]:
summary_csv = OUT_DIR / "summary.csv"
if all_summaries:
    with open(summary_csv, "w", newline="") as f:
        fieldnames = sorted(set().union(*[row.keys() for row in all_summaries]))
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in all_summaries:
            flat_row = {}
            for key, value in row.items():
                flat_row[key] = "|".join(map(str, value)) if isinstance(value, (list, tuple)) else str(value)
            writer.writerow(flat_row)

summary_df = pd.DataFrame(all_summaries)
summary_df


,lambda_pde,seed,final_loss,final_loss_data,final_loss_pde,l1,loss_tv,pde_method,feature_names,coefficients,...,num_samples,u_rel_l2,u_rmse,u_max_abs,ux_rel_l2,ux_rmse,ux_max_abs,uxx_rel_l2,uxx_rmse,uxx_max_abs
0,0.1,0,1.589049,0.756189,8.299707,2.889066,0.0,least_squares,"[u, u_x, u_xx]","[0.002679550167974973, -0.0026570879188046355,...",...,128,0.980216,0.851192,2.434150,7.907529,8.245793,25.807659,32.058653,170.316028,520.729302
1,0.3,0,4.472592,1.173864,10.988905,2.055610,0.0,least_squares,"[u, u_x, u_xx]","[-0.3869209250193535, 0.01924412408828383, 0.0...",...,128,1.235389,1.072777,2.917726,9.692242,10.106852,24.966440,42.701934,226.859925,1007.155292
2,0.5,0,5.041625,1.263301,7.550605,3.022402,0.0,least_squares,"[u, u_x, u_xx]","[-0.17894489116613216, -0.02805963632917832, -...",...,128,1.282346,1.113554,2.846753,8.597235,8.965003,27.643400,39.305132,208.813948,623.154363
3,0.7,0,6.633024,1.167032,7.804345,2.950827,0.0,least_squares,"[u, u_x, u_xx]","[-0.31989249364443195, 0.015909027296061213, -...",...,128,1.235891,1.073214,2.335850,7.493306,7.813852,22.540906,31.900038,169.473360,524.148391


## 7. Inspect Saved Artifacts

Use this section to quickly browse output folders and important metrics after a run.


In [7]:
sorted((OUT_DIR / "burgers").glob("lambda_*/*"))[:10]


[WindowsPath('c:/Users/sami/OneDrive/Documents/PDENets/run_results/pde_lambda_sweep_notebook/burgers/lambda_0.1/seed_000'),
 WindowsPath('c:/Users/sami/OneDrive/Documents/PDENets/run_results/pde_lambda_sweep_notebook/burgers/lambda_0.3/seed_000'),
 WindowsPath('c:/Users/sami/OneDrive/Documents/PDENets/run_results/pde_lambda_sweep_notebook/burgers/lambda_0.5/seed_000'),
 WindowsPath('c:/Users/sami/OneDrive/Documents/PDENets/run_results/pde_lambda_sweep_notebook/burgers/lambda_0.7/seed_000')]

In [8]:
metric_cols = [
    col for col in summary_df.columns
    if col.endswith("_rel_l2") or col.endswith("_rmse") or col.endswith("_max_abs")
]
summary_df[["lambda_pde", "seed", "final_loss", "final_loss_pde", *metric_cols]].sort_values(["lambda_pde", "seed"])


,lambda_pde,seed,final_loss,final_loss_pde,u_rel_l2,u_rmse,u_max_abs,ux_rel_l2,ux_rmse,ux_max_abs,uxx_rel_l2,uxx_rmse,uxx_max_abs
0,0.1,0,1.589049,8.299707,0.980216,0.851192,2.434150,7.907529,8.245793,25.807659,32.058653,170.316028,520.729302
1,0.3,0,4.472592,10.988905,1.235389,1.072777,2.917726,9.692242,10.106852,24.966440,42.701934,226.859925,1007.155292
2,0.5,0,5.041625,7.550605,1.282346,1.113554,2.846753,8.597235,8.965003,27.643400,39.305132,208.813948,623.154363
3,0.7,0,6.633024,7.804345,1.235891,1.073214,2.335850,7.493306,7.813852,22.540906,31.900038,169.473360,524.148391


## 8. Next Analysis Steps

Typical follow-ups:

- inspect one run directory's `summary.json`
- open `feature_overlays/*.pdf`
- compare EQL readout terms across lambdas
- extend aggregation to `summary_agg.csv` if you want seed-level reduction in the notebook too
